In [1]:
!pip install pandas nltk gensim

In [2]:
# Name: Muhammad Adli Bin Rosdan
# ID: SW01083749
# Name: Muhammad Irfan Azraei Bin Izhar Kamil
# ID: SW01083594

import pandas as pd
import numpy as np
import re
import nltk
import gensim
import warnings

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer
from gensim import corpora
from gensim.models import CoherenceModel

warnings.filterwarnings("ignore")

In [3]:
# Run this cell once only if NLTK data is missing
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\adli\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\adli\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\adli\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [4]:
# Put news_dataset.csv in the same folder as this notebook
df = pd.read_csv("news_dataset.csv")

# Use only the text column
df = df[['text']]

df.head()

,text
0,I was wondering if anyone out there could enli...
1,I recently posted an article asking what kind ...
2,\nIt depends on your priorities. A lot of peo...
3,an excellent automatic can be found in the sub...
4,: Ford and his automobile. I need information...


In [5]:
# Remove null values
df = df.dropna(subset=['text'])

# Convert to string just to be safe
df['text'] = df['text'].astype(str)

print("Total rows after removing null values:", len(df))

Total rows after removing null values: 11096


In [6]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()
stemmer = PorterStemmer()

def preprocess_text(text):
    # lowercase
    text = text.lower()
    
    # remove punctuation, numbers, special characters
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    
    # tokenize
    tokens = text.split()
    
    # remove stopwords and short words
    tokens = [word for word in tokens if word not in stop_words and len(word) > 2]
    
    # lemmatization
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    
    # stemming
    tokens = [stemmer.stem(word) for word in tokens]
    
    return tokens

df['processed_text'] = df['text'].apply(preprocess_text)

df.head()

,text,processed_text
0,I was wondering if anyone out there could enli...,"[wonder, anyon, could, enlighten, car, saw, da..."
1,I recently posted an article asking what kind ...,"[recent, post, articl, ask, kind, rate, singl,..."
2,\nIt depends on your priorities. A lot of peo...,"[depend, prioriti, lot, peopl, put, higher, pr..."
3,an excellent automatic can be found in the sub...,"[excel, automat, found, subaru, legaci, switch..."
4,: Ford and his automobile. I need information...,"[ford, automobil, need, inform, whether, ford,..."


In [7]:
# Create dictionary and corpus for Gensim
texts = df['processed_text'].tolist()

dictionary = corpora.Dictionary(texts)

# Filter extreme words
dictionary.filter_extremes(no_below=5, no_above=0.5)

corpus = [dictionary.doc2bow(text) for text in texts]

print("Number of documents:", len(corpus))
print("Number of unique tokens:", len(dictionary))

Number of documents: 11096
Number of unique tokens: 10991


In [8]:
# Build LDA model with 4 topics
lda_model = gensim.models.LdaModel(
    corpus=corpus,
    id2word=dictionary,
    num_topics=4,
    random_state=42,
    passes=10,
    alpha='auto',
    per_word_topics=True
)

In [9]:
# Display the 4 topics
for idx, topic in lda_model.print_topics(num_words=10):
    print(f"Topic {idx + 1}:")
    print(topic)
    print()

Topic 1:
0.016*"use" + 0.014*"key" + 0.008*"encrypt" + 0.008*"file" + 0.008*"system" + 0.006*"chip" + 0.006*"program" + 0.005*"bit" + 0.005*"window" + 0.005*"one"

Topic 2:
0.010*"peopl" + 0.008*"would" + 0.007*"one" + 0.006*"govern" + 0.005*"say" + 0.005*"god" + 0.005*"law" + 0.004*"state" + 0.004*"right" + 0.004*"think"

Topic 3:
0.010*"get" + 0.010*"would" + 0.010*"one" + 0.008*"like" + 0.008*"know" + 0.007*"think" + 0.007*"time" + 0.006*"year" + 0.006*"go" + 0.006*"good"

Topic 4:
0.036*"max" + 0.010*"edu" + 0.009*"team" + 0.005*"game" + 0.005*"space" + 0.005*"new" + 0.005*"play" + 0.005*"univers" + 0.005*"com" + 0.005*"leagu"



In [10]:
# Evaluate using coherence score
coherence_model = CoherenceModel(
    model=lda_model,
    texts=texts,
    dictionary=dictionary,
    coherence='c_v'
)

coherence_score = coherence_model.get_coherence()
print("Coherence Score:", coherence_score)

Coherence Score: 0.5463926249962652


In [11]:
# Assign dominant topic to each document
def get_dominant_topic(bow):
    topic_probs = lda_model.get_document_topics(bow)
    return max(topic_probs, key=lambda x: x[1])[0]

df['dominant_topic'] = [get_dominant_topic(doc) for doc in corpus]

df[['text', 'dominant_topic']].head(10)

,text,dominant_topic
0,I was wondering if anyone out there could enli...,2
1,I recently posted an article asking what kind ...,2
2,\nIt depends on your priorities. A lot of peo...,2
3,an excellent automatic can be found in the sub...,2
4,: Ford and his automobile. I need information...,2
5,\nYo! Watch the attributions--I didn't say tha...,2
6,\nYou can avoid these problems entirely by ins...,2
7,"I have a 1986 Acura Integra 5 speed with 95,00...",2
8,"\nassuming yours is a non turbo MR2, the gruff...",2
9,"\n\nIn addition to restricted mileage, many cl...",2


In [12]:
# Count documents in each topic
topic_counts = df['dominant_topic'].value_counts().sort_index()
print(topic_counts)

dominant_topic
0    3228
1    2882
2    4464
3     522
Name: count, dtype: int64


In [13]:
# Optional: save result
df.to_csv("lda_topic_results.csv", index=False)
print("File saved as lda_topic_results.csv")

File saved as lda_topic_results.csv


### Interpretation of Coherence Score

The coherence score is used to measure how meaningful and interpretable the topics generated 
by the LDA model are. In this assignment, the coherence score obtained is 
**0.5463926249962652**. A higher coherence score generally indicates that the words 
grouped under each topic are more related and easier to understand. This suggests that the LDA model is 
reasonably good at identifying meaningful hidden themes from the news text data. Based on the result,
the model shows that the dataset can be divided into 4 main topics, although the quality 
of the topics may still depend on the text preprocessing steps and the number of topics chosen.